<a href="https://colab.research.google.com/github/miriamamin1213-ux/HPV-classification/blob/main/Transformer_model_training_LR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# MODEL 27
# FEATURE TOKEN TRANSFORMER (FTT)
# HPV PREDICTION
# 11 FEATURES
# SEED 42
# ============================================================

!pip install -q gdown

import gdown

gdown.download(
    id="1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv",
    output="HPV2025.xlsx",
    quiet=False
)

# ============================================================
# Imports
# ============================================================

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    roc_auc_score,
    f1_score
)

from imblearn.over_sampling import SMOTE

# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

SEED = 42

seed_everything(SEED)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device :", device)

# ============================================================
# Read Dataset
# ============================================================

df = pd.read_excel("HPV2025.xlsx")

df = df.dropna(subset=["HPV Status"])

# ============================================================
# Missing Values
# ============================================================

df["Tobacco Consumption"] = df[
    "Tobacco Consumption"
].fillna(
    df["Tobacco Consumption"].mode()[0]
)

df["Alcohol Consumption"] = df[
    "Alcohol Consumption"
].fillna(
    df["Alcohol Consumption"].mode()[0]
)

# ============================================================
# Remove Unused Columns
# ============================================================

df = df.drop(
    columns=[
        "PatientID",
        "CenterID",
        "Task 1",
        "Task 2",
        "Task 3"
    ]
)

df = df.dropna()

print()
print("Dataset Shape :", df.shape)

# ============================================================
# Encode Cancer Stage
# ============================================================

df["T-stage"] = df["T-stage"].replace({
    "T0":0,
    "T1":1,
    "T2":2,
    "T3":3,
    "T4":4
})

df["N-stage"] = df["N-stage"].replace({
    "N0":0,
    "N1":1,
    "N2":2,
    "N3":3
})

df["M-stage"] = df["M-stage"].replace({
    "M0":0,
    "M1":1
})

# ============================================================
# Features
# ============================================================

feature_names = [

    "Age",
    "Gender",
    "Tobacco Consumption",
    "Alcohol Consumption",
    "Performance Status",
    "Relapse",
    "RFS",
    "Treatment",
    "T-stage",
    "N-stage",
    "M-stage"

]

X = df[feature_names]
y = df["HPV Status"]

# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

X_train_full, X_test, y_train_full, y_test = train_test_split(

    X,
    y,
    test_size=0.20,
    random_state=SEED

)

# ============================================================
# TRAIN / VALIDATION SPLIT
# ============================================================

X_train, X_val, y_train, y_val = train_test_split(

    X_train_full,
    y_train_full,

    test_size=0.20,

    random_state=SEED,

    stratify=y_train_full

)

print()

print("Training Samples")
print(y_train.value_counts())

print()

print("Validation Samples")
print(y_val.value_counts())

print()

print("Testing Samples")
print(y_test.value_counts())

# ============================================================
# Standardisation
# ============================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_val = scaler.transform(X_val)

X_test = scaler.transform(X_test)

# ============================================================
# SMOTE
# ============================================================

smote = SMOTE(random_state=SEED)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print()

print("After SMOTE")

print(pd.Series(y_train_smote).value_counts())

# ============================================================
# Torch Tensors
# ============================================================

X_train_tensor = torch.tensor(
    X_train_smote,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_smote.values,
    dtype=torch.long
)

y_val_tensor = torch.tensor(
    y_val.values,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test.values,
    dtype=torch.long
)

# ============================================================
# DataLoader
# ============================================================

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print()

print("Training Shape :", X_train_tensor.shape)

print("Validation Shape :", X_val_tensor.shape)

print("Testing Shape :", X_test_tensor.shape)

print()

print("Preprocessing Complete.")

# ============================================================
# MODEL 27
# FEATURE TOKEN TRANSFORMER (FTT)
# ============================================================

class FeatureTokenTransformer(nn.Module):

    def __init__(

        self,

        num_features=11,
        d_model=32,
        nhead=4,
        num_layers=2,
        num_classes=2,
        dropout=0.20

    ):

        super().__init__()

        self.num_features = num_features
        self.d_model = d_model

        # ----------------------------------------------------
        # Feature Token Embeddings
        # ----------------------------------------------------

        self.feature_embeddings = nn.ModuleList([

            nn.Linear(1, d_model)

            for _ in range(num_features)

        ])

        # ----------------------------------------------------
        # Learnable Positional Embeddings
        # ----------------------------------------------------

        self.position_embedding = nn.Parameter(

            torch.randn(1, num_features, d_model)

        )

        # ----------------------------------------------------
        # Transformer Encoder
        # ----------------------------------------------------

        encoder_layer = nn.TransformerEncoderLayer(

            d_model=d_model,
            nhead=nhead,
            dim_feedforward=128,
            dropout=dropout,
            activation="gelu",
            batch_first=True

        )

        self.transformer = nn.TransformerEncoder(

            encoder_layer,
            num_layers=num_layers

        )

        # ----------------------------------------------------
        # Classification Head
        # ----------------------------------------------------

        self.flatten = nn.Flatten()

        self.norm = nn.LayerNorm(

            num_features * d_model

        )

        self.fc1 = nn.Linear(

            num_features * d_model,
            32

        )

        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(

            dropout

        )

        self.fc2 = nn.Linear(

            32,
            num_classes

        )

    # ========================================================
    # Forward
    # ========================================================

    def forward(self, x):

        tokens = []

        for i in range(self.num_features):

            token = self.feature_embeddings[i](

                x[:, i:i+1]

            )

            tokens.append(token)

        x = torch.stack(

            tokens,
            dim=1

        )

        # Positional Embeddings

        x = x + self.position_embedding

        # Transformer Encoder

        x = self.transformer(x)

        # Classification Head

        x = self.flatten(x)

        x = self.norm(x)

        x = self.relu(

            self.fc1(x)

        )

        x = self.dropout(x)

        x = self.fc2(x)

        return x


# ============================================================
# Build Model
# ============================================================

model = FeatureTokenTransformer(

    num_features=11,
    d_model=32,
    nhead=4,
    num_layers=2,
    num_classes=2,
    dropout=0.20

).to(device)

# ============================================================
# Loss Function
# ============================================================

class_weights = torch.tensor(

    [2.0, 1.0],
    dtype=torch.float32

).to(device)

criterion = nn.CrossEntropyLoss(

    weight=class_weights

)

# ============================================================
# Optimiser
# ============================================================

optimizer = optim.Adam(

    model.parameters(),
    lr=0.001

)

print(model)

print()

print("Transformer Created Successfully")

# ============================================================
# TRAINING
# WITH VALIDATION + EARLY STOPPING
# ============================================================

num_epochs = 300

patience = 20
best_val_loss = float("inf")
early_stop_counter = 0

train_losses = []
val_losses = []

print()
print("Starting Training...\n")

for epoch in range(num_epochs):

    # ========================================================
    # TRAINING
    # ========================================================

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)

    train_acc = 100 * correct / total

    train_losses.append(train_loss)

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_running_loss = 0.0

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for inputs, labels in val_loader:

            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)

            loss = criterion(outputs, labels)

            val_running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()

    val_loss = val_running_loss / len(val_loader)

    val_acc = 100 * val_correct / val_total

    val_losses.append(val_loss)

    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        early_stop_counter = 0

        torch.save(

            model.state_dict(),
            "best_transformer_model.pth"

        )

    else:

        early_stop_counter += 1

    # ========================================================
    # PRINT PROGRESS
    # ========================================================

    if (epoch + 1) % 10 == 0:

        print(

            f"Epoch {epoch+1:3d}/{num_epochs}"

            f" | Train Loss: {train_loss:.4f}"

            f" | Val Loss: {val_loss:.4f}"

            f" | Train Acc: {train_acc:.2f}%"

            f" | Val Acc: {val_acc:.2f}%"

        )

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if early_stop_counter >= patience:

        print()

        print("Early stopping triggered.")

        print(f"Training stopped at epoch {epoch+1}")

        break

print()

print("Training Complete.")

print()

print(f"Best Validation Loss : {best_val_loss:.6f}")

print()

print("Best model saved as")

print("best_transformer_model.pth")

Downloading...
From: https://drive.google.com/uc?id=1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv
To: /content/HPV2025.xlsx
100%|██████████| 66.0k/66.0k [00:00<00:00, 22.0MB/s]


Device : cpu

Dataset Shape : (423, 12)

Training Samples
HPV Status
1.0    256
0.0     14
Name: count, dtype: int64

Validation Samples
HPV Status
1.0    64
0.0     4
Name: count, dtype: int64

Testing Samples
HPV Status
1.0    80
0.0     5
Name: count, dtype: int64

After SMOTE
HPV Status
1.0    256
0.0    256
Name: count, dtype: int64

Training Shape : torch.Size([512, 11])
Validation Shape : torch.Size([68, 11])
Testing Shape : torch.Size([85, 11])

Preprocessing Complete.


/tmp/ipykernel_2212/1269425629.py:122: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["T-stage"] = df["T-stage"].replace({
/tmp/ipykernel_2212/1269425629.py:130: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["N-stage"] = df["N-stage"].replace({
/tmp/ipykernel_2212/1269425629.py:137: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no

FeatureTokenTransformer(
  (feature_embeddings): ModuleList(
    (0-10): 11 x Linear(in_features=1, out_features=32, bias=True)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
        (linear1): Linear(in_features=32, out_features=128, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=128, out_features=32, bias=True)
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (norm): LayerNorm((352,), eps=1e-05, elementwise_affine=True)
  (fc1): Linear(in_features=352, out_features=32, bias=

In [2]:
# ============================================================
# MODEL EVALUATION
# FEATURE TOKEN TRANSFORMER (FTT)
# ============================================================

# ============================================================
# Load Best Model
# ============================================================

model.load_state_dict(

    torch.load(

        "best_transformer_model.pth",
        map_location=device

    )

)

model.eval()

predictions = []
probabilities = []
actual = []

# ============================================================
# Testing
# ============================================================

with torch.no_grad():

    for inputs, labels in test_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        probs = torch.softmax(outputs, dim=1)

        _, preds = torch.max(outputs, 1)

        predictions.extend(

            preds.cpu().numpy()

        )

        probabilities.extend(

            probs[:,1].cpu().numpy()

        )

        actual.extend(

            labels.cpu().numpy()

        )

predictions = np.array(predictions)
probabilities = np.array(probabilities)
actual = np.array(actual)

# ============================================================
# Classification Report
# ============================================================

print()

print("Classification Report\n")

print(

    classification_report(

        actual,
        predictions,
        digits=4

    )

)

# ============================================================
# Confusion Matrix
# ============================================================

cm = confusion_matrix(

    actual,
    predictions

)

print("Confusion Matrix")

print(cm)

# ============================================================
# Metrics
# ============================================================

accuracy = (

    predictions == actual

).mean()

bal_acc = balanced_accuracy_score(

    actual,
    predictions

)

f1 = f1_score(

    actual,
    predictions

)

auc = roc_auc_score(

    actual,
    probabilities

)

print()

print(f"Accuracy:            {accuracy:.4f}")

print(f"Balanced Accuracy:   {bal_acc:.4f}")

print(f"F1-score:            {f1:.4f}")

print(f"AUC:                 {auc:.4f}")

# ============================================================
# MODEL SUMMARY
# ============================================================

print()

print("====================================")

print("FEATURE TOKEN TRANSFORMER SUMMARY")

print("====================================")

print("Model : Feature Token Transformer")

print("Embedding Size : 32")

print("Attention Heads : 4")

print("Transformer Layers : 2")

print("Dropout : 0.20")

print("Learning Rate : 0.001")

print("Batch Size : 32")

print("Epochs Trained :", epoch + 1)

print("Early Stopping : Yes")

print("Best Validation Loss :", round(best_val_loss, 6))

print("SMOTE : Yes")

print("Training Patients :", len(y_train_tensor))

print("Validation Patients :", len(y_val_tensor))

print("Test Patients :", len(actual))

print()

print("Final Results")

print("---------------------------")

print(f"Accuracy            : {accuracy:.4f}")

print(f"Balanced Accuracy   : {bal_acc:.4f}")

print(f"F1-score            : {f1:.4f}")

print(f"AUC                 : {auc:.4f}")

print()

print("Confusion Matrix")

print(cm)

print()

print("Analysis Complete.")


Classification Report

              precision    recall  f1-score   support

           0     0.2000    0.8000    0.3200         5
           1     0.9846    0.8000    0.8828        80

    accuracy                         0.8000        85
   macro avg     0.5923    0.8000    0.6014        85
weighted avg     0.9385    0.8000    0.8497        85

Confusion Matrix
[[ 4  1]
 [16 64]]

Accuracy:            0.8000
Balanced Accuracy:   0.8000
F1-score:            0.8828
AUC:                 0.9200

FEATURE TOKEN TRANSFORMER SUMMARY
Model : Feature Token Transformer
Embedding Size : 32
Attention Heads : 4
Transformer Layers : 2
Dropout : 0.20
Learning Rate : 0.001
Batch Size : 32
Epochs Trained : 27
Early Stopping : Yes
Best Validation Loss : 0.189294
SMOTE : Yes
Training Patients : 512
Validation Patients : 68
Test Patients : 85

Final Results
---------------------------
Accuracy            : 0.8000
Balanced Accuracy   : 0.8000
F1-score            : 0.8828
AUC                 : 0.9200

Co